# Image Metrics and Radar Plot
### v1.1

*Dr. Avital Wagner* \
*Luco Buise*

**RadboudUMC EMC**

30-06-2026

---

In [ ]:
import torch
import inspect

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from pathlib import Path

import ipywidgets as widgets
from IPython.display import display
# ---- OWN FILES:

#from utils.PIQ_utils import ...
import utils.PIQ_metrics as PIQ_metrics
from utils.PIQ_utils import load_image, df_all_images
from utils.PIQ_plotting import make_radar_plot

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}

## Determine Image Paths

Build your directories using the names below, or adjust them as necessary. You need to have one directory with the images to be assessed, and one with the (single) reference image.

In [ ]:
img_dir = "img_dir" # <---- where are your images located
ref_dir = "ref_dir" # <---- where are your reference images (for SSIM) located

In [ ]:
img_dir = Path(img_dir)
imgs = [img.resolve() for img in img_dir.iterdir() if img.is_file() and img.suffix.lower() in IMG_EXTS] # resolve() for abs path
print("Images found:", imgs)

In [ ]:
# FOR SSIM
# can only do this for one reference
ref_dir = Path(ref_dir)
ref_imgs = [img for img in ref_dir.iterdir() if img.is_file() and img.suffix.lower() in IMG_EXTS]
ref_img, _ = load_image(ref_imgs[0]) # only load first file found (might want to adapt this)

## Metric selection

In [ ]:
# find all classes that inherit from Metric()
metric_classes = {
    name: cls
    for name, cls in inspect.getmembers(PIQ_metrics, inspect.isclass)
    if issubclass(cls, PIQ_metrics.Metric) and cls is not PIQ_metrics.Metric
}

In [ ]:
checkboxes = {
    name: widgets.Checkbox(value=False, description=name)
    for name in sorted(metric_classes)
}

display(widgets.VBox(list(checkboxes.values())))

In [ ]:
metrics = []

# get selected metrics
for name, checkbox in checkboxes.items():
    if checkbox.value: # if selected, create
        metric = metric_classes[name]()
        
        if metric.need_ref_img: # check if it needs a reference image (now only SSIM)
            metric.set_ref(ref_img)

        metrics.append(metric)    

## Adding you own metrics

If you want to use your own metrics, you can add them in the PIQ_metrics.py file. If you just have one, you can also implement it below, following the given structure.

In [ ]:
class YourMetric(PIQ_metrics.Metric):

    def __init__(self):
        super().__init__() 
    
    def name(self):
        return "Your Metric" # <----------- change the name to match your new metric

    def compute(self, img, **kwargs):
        value = ... # <--------------- the metric value should be saved in here
        return {self.name() : value}

metrics.append(YourMetric())

## Hyperparameters and Metric Calculation

In [ ]:
# decide on your hyperparameters
hyperparameters = {
    "nbins" : 300, # FFT
    "tail_frac" : 0.30, # FFT
    "exclude_zero_fft_pixels" : False, # FFT 
    "roughness_perc" : 0.90, # Roughness
    "roughness_sigma" : 1.0, # Roughness
    "SP_window" : 11, # Salt & Pepper
    "SP_threshold" : 0.03, # Salt & Pepper
}

In [ ]:
# calculate all metrics for all images and save in a dataframe
df = df_all_images(imgs, metrics, hyperparameters)

## Saving as CSV

In [ ]:
# adjust ranges as necessary
ranges = {
    "NIQE": "50-0",
    "res_ampl_nm": "010-0.74",
    "PIQE": "0-1.1",
    "Roughness 0.9": "0.02-0",
    "Salt & Pepper Noise": "001-0",
    "SSIM": "0-0.6",
    "BRISQUE": "70-0",
    "DOM": "0-1.2",
    "Your Metric": "0-100",
}

range_row = {
    "image": "",
    **ranges
}

out = pd.concat([pd.DataFrame([range_row]), df], ignore_index=True) # add the ranges to the df
out = out[df.columns.intersection(range_row.keys())] # remove all columns that do not overlap between the df and ranges

In [ ]:
# rename any column you would like
out = out.rename(columns={
    "res_ampl_nm": "FFT Resolution",
})

In [ ]:
# save the dataframe to a csv file
f_name = "metrics" 
out.to_csv(f_name + ".csv", index=False)

---
## Create and Save Radial Plot

In [ ]:
metrics_name = "metrics"

make_radar_plot(metrics_name + ".csv")